In [1]:
import numpy as np # for numerical calculations such as histogramming
import matplotlib.pyplot as plt # for plotting
import matplotlib_inline # to edit the inline plot format
from matplotlib.ticker import AutoMinorLocator # for minor ticks
import uproot # for reading .root files
import awkward as ak # to represent nested data in columnar format
import vector # for 4-momentum calculations
import time # for printing time stamps
import requests # for file gathering, if needed; for HTTP access


/home/aegis/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "


In [10]:
lumi = 36.6 # in fb-1 # data size of the full release

In [11]:
import atlasopenmagic as atom
atom.available_releases()

Available releases:
2016e-8tev           2016 Open Data for education release of 8 TeV proton-proton collisions (https://opendata.cern.ch/record/3860).
2020e-13tev          2020 Open Data for education release of 13 TeV proton-proton collisions (https://cern.ch/2r7xt).
2024r-pp             2024 Open Data for research release for proton-proton collisions (https://opendata.cern.record/80020).
2024r-hi             2024 Open Data for research release for heavy-ion collisions (https://opendata.cern.ch/record/80035).
2025e-13tev-beta     2025 Open Data for education and outreach beta release for 13 TeV proton-proton collisions (https://opendata.cern.ch/record/93910).
2025r-evgen-13tev    2025 Open Data for research release for event generation at 13 TeV (https://opendata.cern.ch/record/160000).
2025r-evgen-13p6tev  2025 Open Data for research release for event generation at 13.6 TeV (https://opendata.cern.ch/record/160000).


{'2016e-8tev': '2016 Open Data for education release of 8 TeV proton-proton collisions (https://opendata.cern.ch/record/3860).',
 '2020e-13tev': '2020 Open Data for education release of 13 TeV proton-proton collisions (https://cern.ch/2r7xt).',
 '2024r-pp': '2024 Open Data for research release for proton-proton collisions (https://opendata.cern.record/80020).',
 '2024r-hi': '2024 Open Data for research release for heavy-ion collisions (https://opendata.cern.ch/record/80035).',
 '2025e-13tev-beta': '2025 Open Data for education and outreach beta release for 13 TeV proton-proton collisions (https://opendata.cern.ch/record/93910).',
 '2025r-evgen-13tev': '2025 Open Data for research release for event generation at 13 TeV (https://opendata.cern.ch/record/160000).',
 '2025r-evgen-13p6tev': '2025 Open Data for research release for event generation at 13.6 TeV (https://opendata.cern.ch/record/160000).'}

In [12]:
atom.set_release("2024r-pp")

Release '2024r-pp' already active with cached metadata.
Active release: 2024r-pp. (Datasets path: REMOTE)


In [30]:
atom.available_datasets()

['301204',
 '301209',
 '301243',
 '301247',
 '301333',
 '301826',
 '301928',
 '302520',
 '302521',
 '302522',
 '302523',
 '302524',
 '302525',
 '302526',
 '302527',
 '302528',
 '302529',
 '302530',
 '302531',
 '302532',
 '302533',
 '302534',
 '302733',
 '304014',
 '306149',
 '311490',
 '312613',
 '341456',
 '341458',
 '341460',
 '343981',
 '344158',
 '345056',
 '345058',
 '345060',
 '345061',
 '345066',
 '345097',
 '345098',
 '345103',
 '345104',
 '345105',
 '345106',
 '345112',
 '345114',
 '345120',
 '345121',
 '345122',
 '345123',
 '345124',
 '345125',
 '345211',
 '345212',
 '345213',
 '345214',
 '345215',
 '345216',
 '345217',
 '345218',
 '345219',
 '345316',
 '345317',
 '345318',
 '345319',
 '345320',
 '345321',
 '345322',
 '345324',
 '345325',
 '345433',
 '345445',
 '345596',
 '345697',
 '345698',
 '345699',
 '345833',
 '345834',
 '345876',
 '345877',
 '345878',
 '345944',
 '345948',
 '345949',
 '345961',
 '345963',
 '345964',
 '345965',
 '346188',
 '346189',
 '346190',
 '346191',

In [ ]:
mc_defs = {
    "ttbar": {
        "dids": [
            "410470", "410471"
        ]
    },
    "SingleTop": {
        "dids": [
            "410644", "410645", "410658", "410659" # Not avail: 410646, 410647
        ]
    },
    "Diboson": {
        "dids": [
            "700488", "700489", "700491", "700495" # Not avail: 363355, 363357, 363359, 363360, 363489
        ]
    },
    "Z+jets": {
        "dids": [
            "700323", "700324", "700325", 
            "700335", "700336", "700337" # Not avail: 700326, 700327, 700328, 700329, 700330, 700331, 700332, 700333, 700334
        ]
    },
    "W+jets": {
        "dids": [
            "700338", "700339", "700340", "700341", "700342", "700343", 
            "700344", "700345", "700346", "700347", "700348", "700349"
        ]
    },
    "Multijet": {
        "dids": [
            "364701", "364702", "364703", "364704", "364705", "364706", 
            "364707", "364708", "364709", "364710", "364711", "364712"
        ]
    }
}

mc_samples   = atom.build_dataset(mc_defs, protocol='https', cache=True)

In [43]:
mc_samples.keys()

dict_keys(['ttbar', 'SingleTop', 'Diboson', 'Z+jets', 'W+jets', 'Multijet'])

In [16]:
all_data = atom.get_urls('data')

In [17]:
# Assuming your list of URLs is stored in a variable called 'urls'

unique_campaigns = set()

for url in all_data:
    # Split by '/' and grab the second-to-last element
    # Example: '.../rucio/data15_13TeV/DAOD...' -> 'data15_13TeV'
    campaign_tag = url.split('/')[-2]
    
    unique_campaigns.add(campaign_tag)

# Convert the set to a sorted list and print
for campaign in sorted(list(unique_campaigns)):
    print(campaign)

data15_13TeV
data16_13TeV


In [48]:
test_files = mc_samples['ttbar']['list'][0]

tree = uproot.open(test_files+":analysis")

FSTimeoutError: 